# 古い対話を要約して長い対話を続ける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_dialogue/chat_summary_memory.ipynb)

NLP クックブック 22-06 ／ 対話システム

> GPU ランタイム推奨 (ランタイム → ランタイムのタイプを変更 → T4 GPU)

長い対話で、古いターンを単に捨てると大事な情報を忘れてしまう。捨てる前に要約して、記憶として持ち越したい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。T4 なら 1 応答あたり数秒〜10 秒ほど (GPU メモリ約 7GB)
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def generate(messages, max_new_tokens=100):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

class SummaryChat:
    def __init__(self, system, keep_turns=2):
        self.system, self.summary, self.history, self.keep = system, "", [], keep_turns * 2

    def compress(self):
        """直近 keep_turns ターンより古い部分を、これまでの要約と合わせて要約し直す"""
        old, self.history = self.history[:-self.keep], self.history[-self.keep:]
        dialogue = "\n".join(f"{'ユーザー' if m['role'] == 'user' else 'アシスタント'}: {m['content']}" for m in old)
        prompt = ("次の「これまでの要約」と「対話」から、ユーザーの名前・予定・好みと、既に紹介した場所を漏らさずに、3 文以内の新しい要約を書いてください。"
                  "対話に出てこないことは書かないでください。\n\n"
                  f"# これまでの要約\n{self.summary or '(なし)'}\n\n# 対話\n{dialogue}")
        self.summary = generate([{"role": "user", "content": prompt}], max_new_tokens=120)

    def say(self, text):
        self.history.append({"role": "user", "content": text})
        if len(self.history) > self.keep + 2:
            self.compress()
        system = self.system + (f"\n\n# これまでの対話で分かっていること\n{self.summary}" if self.summary else "")
        reply = generate([{"role": "system", "content": system}] + self.history, max_new_tokens=60)
        self.history.append({"role": "assistant", "content": reply})
        return reply

# 要約を含めて 8 回生成する。T4 で 1〜2 分 (モデルの読み込みを除く)
chat = SummaryChat("あなたは旅行の相談に乗るアシスタントです。1〜2 文で短く答えてください。"
                   "「これまでの対話で分かっていること」は、あなたがユーザーから既に聞いた内容です。覚えているものとして答えてください。")
for user in ["私の名前は田中です。来月、京都に 2 泊で行く予定です。", "混雑が苦手なので、静かな場所が好きです。", "秋におすすめのお寺を 1 つ教えてください。",
             "そこへは京都駅からどう行けばいいですか？", "近くでお昼を食べるなら何がいいですか？", "最後に確認です。私の名前と、私がどんな場所を好むかを言ってください。"]:
    print(f"あなた: {user}\nボット: {chat.say(user)}")
    if chat.summary:
        print(f"   [要約] {chat.summary}")

あなた: 私の名前は田中です。来月、京都に 2 泊で行く予定です。
ボット: 田中さん、来月の京都旅行楽しみですね。どのようなご予定でしょうか？観光やグルメなど、ご希望をお聞かせください。
あなた: 混雑が苦手なので、静かな場所が好きです。
ボット: 了解しました。京都には静かで落ち着いた雰囲気の場所がたくさんあります。例えば、嵐山の竹林や東福寺の通天橋などはいかがでしょうか。どちらも自然豊かで、混雑を避けながらゆっくりと散策していただけます。また、市内の北西部にある鷹峯や嵯峨野エリアもおすすめです。これらの地域には、
あなた: 秋におすすめのお寺を 1 つ教えてください。
ボット: 秋に京都を訪れるなら、紅葉が美しい東福寺はいかがでしょうか。特に、通天橋から眺める渓谷と紅葉のコントラストは非常に美しく、混雑は避けられませんが、それでも見る価値のある景色です。もし混雑を避けたい場合は、同じ東福寺でも臥雲橋や八相の庭など、他のスポット
あなた: そこへは京都駅からどう行けばいいですか？
ボット: 東福寺へは、京都駅からJR奈良線を利用してお越しいただくのが便利です。京都駅からJR奈良線で一駅目の「稲荷駅」で下車し、そこから徒歩で約15分ほどかかります。または、京都駅から市バスを利用する場合、京都駅前バス停から「208号系統」に乗車し
   [要約] 田中さんは来月京都に2泊する予定で、名前は田中です。彼女は混雑を避け、静かな場所を好むため、京都での旅行では観光やグルメの中でも特に静かなスポットや地元のグルメを楽しむことを予定しています。
あなた: 近くでお昼を食べるなら何がいいですか？
ボット: 東福寺の近くでお昼を食べるなら、京都の郷土料理である湯豆腐や、京野菜の天ぷらなどがおすすめです。東福寺から徒歩圏内にあるお店としては、「豆腐料理 奥丹 清水」や「京野菜レストラン ヴェルデュール」などが人気です。特に、「豆腐料理 奥丹 清水」は
   [要約] 田中さんは来月京都に2泊する予定で、名前は田中です。彼女は混雑を避け、静かな場所を好むため、京都での旅行では観光やグルメの中でも特に静かなスポットや地元のグルメを楽しむことを予定しています。
あなた: 最後に確認です。私の名前と、私がどんな場所を好むかを言ってください。
ボット: もちろんです。私は、ユーザーの名前が田

- 直近の数ターンはそのまま残し、それより古い部分だけを要約に置き換える。最近のやりとりは細部まで、昔のことは要点だけ覚えている状態になる。
- 要約は LLM の呼び出しが 1 回増える。毎ターンではなく、履歴が一定の長さを超えたときだけ行う。
- 要約を重ねるうちに細部は落ちていく。確実に覚えておきたい事実 (名前、日程、決定事項) は、要約とは別に項目として抽出して保存する方法もある。
- 過去の対話を検索できる形で保存しておき、関係する部分だけを取り出して入力に加える方法 (RAG と同じ考え方) もよく使われる。
- この例では、最初のターンが要約に置き換わったあとでも、最後の質問に名前 (田中) と好み (静かな場所) を正しく答えた。この章の「対話の履歴を管理して、長くなったら古いターンを捨てる」では、古いターンを捨てたために名前を忘れる。
- 要約にも誤りが入る。「対話に出てこないことは書かない」と指示しても、要約には対話にない「彼女は」(性別) が入った。要約は次の入力にそのまま使われるので、誤りも持ち越される。
- 応答の中身も確かめる必要がある。静かな場所が好きだと言われたのに、混雑する紅葉の名所 (東福寺) を勧め、行き方では「JR 奈良線で一駅目の稲荷駅」と答えた (一駅目は東福寺駅)。3B のモデルが答える地名や店名、経路は誤りを含みうる。